# Security ML Lab — Exploratory Analysis
Connect to Elasticsearch, pull recent events, and explore anomaly scores.

In [ ]:
from elasticsearch import Elasticsearch
import pandas as pd
import plotly.express as px

# Use 'elasticsearch' when running inside Docker, 'localhost' otherwise
client = Elasticsearch('http://localhost:9200')
print(client.info()['version']['number'])

In [ ]:
resp = client.search(
    index='siem-logs',
    query={'match_all': {}},
    size=500,
    sort=[{'timestamp': 'desc'}],
)
df = pd.DataFrame([h['_source'] for h in resp['hits']['hits']])
df['timestamp'] = pd.to_datetime(df['timestamp'])
df.head()

In [ ]:
# Score distribution
df['anomaly_score'] = pd.to_numeric(df['anomaly_score'], errors='coerce')
px.histogram(df, x='anomaly_score', nbins=50, title='Anomaly Score Distribution')

In [ ]:
# Top anomalous source IPs
df[df['anomaly_score'] > 0.75].groupby('source_ip')['anomaly_score'].mean().sort_values(ascending=False).head(20)

In [ ]:
# Run scoring directly from notebook
import sys; sys.path.insert(0, '..')
from src.models.anomaly import run
run(hours=24, method='isolation_forest')